# 3. Model Prototyping (Recall Model)

Giai đoạn này tập trung vào việc:
1. Đọc dữ liệu đã qua tiền xử lý (Parquet).
2. Encode các User ID và Product ID thành index liên tục (0 tới N-1).
3. Xây dựng lớp Dataset và DataLoader của PyTorch.
4. Thiết kế mô hình nhúng (Matrix Factorization) để sinh ứng viên.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. Load Processed Data

In [ ]:
# Tải data đã được gán nhãn ở phase trước
data_path = '../data/sessions/labeled_sessions.parquet'
df = pd.read_parquet(data_path)
print(f"Data shape: {df.shape}")
df.head()

### 2. Label Encoding (Chuyển ID thành Index 0 -> N-1)

In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_idx'] = user_encoder.fit_transform(df['user_id'])
df['item_idx'] = item_encoder.fit_transform(df['product_id'])

num_users = df['user_idx'].nunique()
num_items = df['item_idx'].nunique()

print(f"Number of Users: {num_users}")
print(f"Number of Items: {num_items}")

### 3. PyTorch Dataset & DataLoader

In [ ]:
class ImplicitFeedbackDataset(Dataset):
    def __init__(self, users, items, labels):
        self.users = torch.tensor(users.values, dtype=torch.long)
        self.items = torch.tensor(items.values, dtype=torch.long)
        self.labels = torch.tensor(labels.values, dtype=torch.float32)
        
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

dataset = ImplicitFeedbackDataset(df['user_idx'], df['item_idx'], df['label'])
# Để batch_size lớn trên GPU giúp train nhanh hơn
dataloader = DataLoader(dataset, batch_size=4096, shuffle=True, num_workers=0)

print(f"Total batches: {len(dataloader)}")

### 4. Matrix Factorization Model (Recall Stage)

In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super(MatrixFactorization, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        # Initialize weights properly
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        
    def forward(self, user_idx, item_idx):
        u_emb = self.user_embedding(user_idx)
        i_emb = self.item_embedding(item_idx)
        
        # Tính Dot Product cho dự đoán
        dot_product = (u_emb * i_emb).sum(dim=1)
        # Trả về xác suất bằng Sigmoid vì nhãn là 0/1
        return torch.sigmoid(dot_product)

### 5. Training Loop Thử nghiệm

In [ ]:
model = MatrixFactorization(num_users, num_items, embedding_dim=32).to(device)
# Vì dữ liệu label là [0, 1] giả nhị phân, ta dùng Binary Cross Entropy
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    
    for users, items, labels in dataloader:
        users, items, labels = users.to(device), items.to(device), labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(users, items)
        
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")